## Task 1

Protein-protein interaction networks can have many different purposes. For example, they can show us if different transcription factors act on the same targets. TFLink (https://tflink.net/)  uniquely provides comprehensive and highly accurate information on transcription factor - target gene interactions, nucleotide sequences and genomic locations of transcription factor binding sites for human and six model organisms: mouse (Mus musculus), rat (Rattus norvegicus), zebrafish (Danio rerio), fruit fly (Drosophila melanogaster), nematode (Caenorhabditis elegans), and yeast (Saccharomyces cerevisiae). TFLink contains clearly identified data, and provides information about the sources: databases, experimental methods and publications.

**The aim of this task is to create a bipartite network to find TF pairs with the most common target.**

**a) You will have to use the human (Homo sapiens) interaction table including all (small- and large-scale) interactions from the TFLink website .**
The dataset for this assignment can be downloaded from the Download part of the website, but since it is quite large, it is placed to the public folder with read-only permissions so that everyone can access it from there. `/v/courses/2026-dataexplorationandvisualization2026.public/Datasets/Networks/mouse.protein.physical.links.v12.0.txt`

**b) Narrow down the interactions to those that have small-scale evidence. Only select those TFs which have a maximum of 50 targets.**

**c) Determine the common targets of each possible TF pair. Which TF pair has the most common targets?**

**d) Visualize this TF pair and their targets. Try and arrange the network so that the common targets are together. Use different colors to separate TFs and targets. (Hint: You can use the pyvis package for less cluttered visualization.)**

**e) Create network where the nodes, the TFs are connected with a weighted edge! The weight should be the normalized value of the number of common targets.**

<div class="margin-left">
    <a class="anchor" id="faq_what-do-interaction-tables-contain"></a>
    <h4>
      What do interaction tables contain?
    </h4>
    <p class="justified">
      Interaction table files are tab separated tables (<i>TSV</i>) of transcription factor - target
      gene
      interactions that contain either interactions validated by small-scale experiments or
      large-scale experiments or these two data altogether. Interaction tables contain the following
      data:
    </p><ol>
      <li><strong>UniprotID.TF</strong> and/or* <strong>UniprotID.Target:</strong>
        <i>Uniprot IDs</i> of
        transcription factors and/or target genes
      </li>
      <li>
        <strong>NCBI.GeneID.TF</strong> and/or <strong>NCBI.GeneID.Target:</strong> <i>NCBI Gene
          IDs</i> of
        transcription factors and/or target genes,
      </li>
      <li><strong>Name.TF</strong> and/or <strong>Name.Target:</strong> gene names of transcription
        factors and/or target genes,</li>
      <li><strong>Detection.method:</strong> names of the detection methods,</li>
      <li><strong>PubmedID:</strong> <i>Pubmed IDs</i> of the original publications (when available)
        and the publications of the databases,</li>
      <li><strong>Organism:</strong> scientific name of the organism,</li>
      <li><strong>Source.database:</strong> names of the original source databases, and</li>
      <li>
        <strong>Small-scale.evidence:</strong> indication about if the data were confirmed by
        small-scale evidence (with "Yes" or "No").
      </li>
      <li>
        <strong>TF.TFLink.ortho:</strong> <i>UniProt IDs</i> of ortholog transcription factors that
        are available at the TFLink gateway. Each entry consists of a shortened name of the organism
        (Hs: <i>Homo sapiens</i>,
        Mm: <i>Mus musculus</i>, Rn: <i>Rattus norvegicus</i>, Dr: <i>Danio rerio</i>, Dm:
        <i>Drosophila
          melanogaster</i>, Ce:
        <i>Caenorhabditis elegans</i>, Sc: <i>Saccharomyces cerevisiae</i>) and the UniProt ID
        separated by a colon (<i>e.g.</i> Mm:Q3UPW2).
      </li>
      <li>
        <strong>TF.nonTFLink.ortho:</strong> <i>UniProt IDs</i> of ortholog transcription factors
        that are not available at the TFLink gateway.
      </li>
      <li>
        <strong>Target.TFLink.ortho:</strong> <i>UniProt IDs</i> of ortholog target genes that are
        available at the TFLink gateway.
      </li>
      <li>
        <strong>Target.nonTFLink.ortho:</strong> <i>UniProt IDs</i> of ortholog target genes that
        are not available at the TFLink gateway.
      </li>
    </ol>
    <hr style="height: 1px;">
    * If the interaction table was downloaded at the `Download` section the file contains
    both TF and
    <i>Target IDs</i> and names. If it was downloaded from an entry page, it contains names and IDs
    of the
    TF (when downloading "Transcription factors of …" or names and IDs of the targets (when
    downloading "Targets of …")
    <p></p>
  </div>

In [2]:
import numpy as np
import os
import networkx as nx
import networkx.algorithms.community as nx_comm
import pandas as pd

import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from dotenv import load_dotenv
load_dotenv("env_variables")
DATASET_DIR = os.getenv('DATASET_DIR')

In [ ]:
# Download the dataset if not using Kooplex
# IT IS ALREADY DOWNLOADED IN THE DATASET_DIR IN KOOPLEX
# !wget https://cdn.netbiol.org/tflink/download_files/TFLink_Homo_sapiens_interactions_All_simpleFormat_v1.0.tsv.gz

--2026-02-24 12:49:23--  https://cdn.netbiol.org/tflink/download_files/TFLink_Homo_sapiens_interactions_All_simpleFormat_v1.0.tsv.gz
Resolving cdn.netbiol.org (cdn.netbiol.org)... 104.18.42.227, 172.64.145.29, 2606:4700:4405::ac40:911d, ...
Connecting to cdn.netbiol.org (cdn.netbiol.org)|104.18.42.227|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 309414866 (295M) [application/gzip]
Saving to: ‘TFLink_Homo_sapiens_interactions_All_simpleFormat_v1.0.tsv.gz’

TFLink_Homo_sapiens 100%[===================>] 295,08M  85,2MB/s    in 3,8s    

2026-02-24 12:49:27 (77,6 MB/s) - ‘TFLink_Homo_sapiens_interactions_All_simpleFormat_v1.0.tsv.gz’ saved [309414866/309414866]



In [4]:
## a)
## read TFlink database entry containing all human interactions

df = pd.read_csv(os.path.join(DATASET_DIR,'TFLink_Homo_sapiens_interactions_All_simpleFormat_v1.0.tsv'),
delimiter='\t',
nrows=10000)
#df = pd.read_csv(os.path.join(DATASET_DIR,'TFLink_Homo_sapiens_interactions_All_simpleFormat_v1.0.tsv'),
#                delimiter='\t')

df.head()

FileNotFoundError: [Errno 2] No such file or directory: './TFLink_Homo_sapiens_interactions_All_simpleFormat_v1.0.tsv'

### b) Narrow down the interactions to those that have small-scale evidence. Only select those TFs which have a maximum of 50 targets.

### c) Determine the common targets of each possible TF pair. Which TF pair has the most common targets?

### d) Visualize this TF pair and their targets. Try and arrange the network so that the common targets are together. Use different colors to separate TFs and targets. (Hint: You can use the pyvis package for less cluttered visualization.)

In [ ]:
#!pip install pyvis

In [16]:
from pyvis.network import Network

### e) Create network where the nodes, the TFs are connected with a weighted edge! The weight should be the normalized value of the number of common targets.